In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, InMemoryDataset, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from rdkit import Chem


# ---------- SMILES -> グラフ変換 ----------
def smiles_to_graph(smiles, y=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # ノード特徴量（原子番号, 価数, 芳香族フラグ など簡易版）
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic()),
            atom.GetTotalNumHs(),
        ])
    x = torch.tensor(atom_features, dtype=torch.float)

    # エッジ（無向グラフなので両方向に追加）
    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
    if len(edge_index) == 0:
        return None
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    data = Data(x=x, edge_index=edge_index)
    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)
    data.smiles = smiles
    return data


# ---------- Dataset ----------
class QM9GapDataset(InMemoryDataset):
    def __init__(self, df, has_label=True):
        super().__init__(None)
        data_list = []
        for _, row in df.iterrows():
            y = row["gap_eV"] if has_label else None
            g = smiles_to_graph(row["smiles"], y)
            if g is not None:
                data_list.append(g)
        self.data, self.slices = self.collate(data_list)


# ---------- GNNモデル ----------
class GCNRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        x = self.lin2(x)
        return x.squeeze(-1)


# ---------- データ読み込み ----------
train_df = pd.read_csv("for_M1/qm9_bandgap_train.csv")   # columns: smiles, gap_eV
test_df = pd.read_csv("for_M1/qm9_bandgap_test_without_answer.csv")  # columns: smiles

train_dataset = QM9GapDataset(train_df, has_label=True)
test_dataset = QM9GapDataset(test_df, has_label=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCNRegressor(in_dim=train_dataset.num_node_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()  # MAEをそのまま最適化


# ---------- 学習 ----------
def train_one_epoch():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(pred, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_dataset)


n_epochs = 100
for epoch in range(1, n_epochs + 1):
    train_mae = train_one_epoch()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train MAE: {train_mae:.4f}")


# ---------- 推論 & submission.csv作成 ----------
model.eval()
smiles_list = []
pred_list = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.batch)
        smiles_list.extend(batch.smiles)
        pred_list.extend(pred.cpu().tolist())

submission = pd.DataFrame({"smiles": smiles_list, "gap_eV": pred_list})
submission.to_csv("submission.csv", index=False)
print("submission.csv を作成しました")

/var/folders/q4/lfsmj5g15fv1xrzvcd566k640000gp/T/ipykernel_80613/166331780.py:84: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
/var/folders/q4/lfsmj5g15fv1xrzvcd566k640000gp/T/ipykernel_80613/166331780.py:85: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


Epoch  10 | Train MAE: 0.7571
Epoch  20 | Train MAE: 0.7534
Epoch  30 | Train MAE: 0.7479
Epoch  40 | Train MAE: 0.6870
Epoch  50 | Train MAE: 0.5592
Epoch  60 | Train MAE: 0.5099
Epoch  70 | Train MAE: 0.4947
Epoch  80 | Train MAE: 0.4561
Epoch  90 | Train MAE: 0.4533
Epoch 100 | Train MAE: 0.4379
submission.csv を作成しました
